# Corpus-regression MaxRL

In [1]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.corpus_regression.maxrl import CorpusRegressionMaxRLConfig
from src.experiments.corpus_regression.utils import dim_averaged_metrics_from_parquet

repo_root = get_repo_base()
device = torch.device("cuda:0")

/home/nlyu/Code/maxrl-statistics/src/experiments/corpus_regression/maxrl.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


### Configs

In [2]:
config = CorpusRegressionMaxRLConfig.get_canonical(
    dataset_base_folder=repo_root / "artifacts" / "corpus-regression",
    study_base_folder=repo_root / "artifacts" / "corpus-regression-maxrl-example",
    num_lookforward_tokens=4,
    train_epochs=2,
    num_rollouts_per_sample=128,
    gaussian_stdev=1.0,
    subtract_baseline=True,
    use_factorized_likelihoods=True,
)

display(config.visualize())

/home/nlyu/Code/maxrl-statistics/src/experiments/corpus_regression/config.py:163: UserWarning: /home/nlyu/Code/maxrl-statistics/artifacts/corpus-regression-maxrl-example/fineweb_edu_smollm2_135m_50000x128_look4_dim32/config.json already exists with different config contents
  warnings.warn(


In [3]:
state = config.initialize(device=device)
state.run_training()

/home/nlyu/Code/maxrl-statistics/src/experiments/corpus_regression/config.py:163: UserWarning: /home/nlyu/Code/maxrl-statistics/artifacts/corpus-regression-maxrl-example/fineweb_edu_smollm2_135m_50000x128_look4_dim32/config.json already exists with different config contents
  warnings.warn(


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

maxrl epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 0:   0%|          | 0/390 [00:00<?, ?it/s]

maxrl epoch 1:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 1:   0%|          | 0/390 [00:00<?, ?it/s]

### Results

In [4]:
metrics = pl.read_parquet(config.study_folder / "metrics.parquet")
metrics

epoch,train_target_xx,train_target_xy,train_target_yy,train_target_n,val_target_xx,val_target_xy,val_target_yy,val_target_n
i64,list[f64],list[f64],list[f64],f64,list[f64],list[f64],list[f64],f64
0,"[3682.428711, 3359.152588, … 3187.191895]","[498.529877, 510.017944, … 186.431717]","[49984.0, 49984.0, … 49984.0]",49984.0,"[107.220245, 609.318115, … 557.580933]","[41.114578, -467.250763, … -228.176666]","[49920.0, 49920.0, … 49920.0]",49920.0
1,"[1781.832153, 1840.870605, … 1559.083374]","[799.783264, 1025.13916, … 663.724121]","[49984.0, 49984.0, … 49984.0]",49984.0,"[3148.568115, 1130.870361, … 564.549377]","[1195.406616, -236.757187, … 32.896568]","[49920.0, 49920.0, … 49920.0]",49920.0


In [5]:
dim_averaged_metrics_from_parquet(metrics, split="train").join(
    dim_averaged_metrics_from_parquet(metrics, split="val"), on="epoch"
)

epoch,train_avg_corr,train_avg_rsq,val_avg_corr,val_avg_rsq
i64,f64,f64,f64,f64
0,0.037843,-0.045363,0.052532,-0.017401
1,0.09386,0.001498,0.032604,-0.016941


In [6]:
last_epoch = int(metrics["epoch"].max())
validation_df = pl.read_parquet(
    config.study_folder / str(last_epoch) / "validation.parquet"
)
validation_df.head()

model_preds,target
list[f64],list[f64]
"[-0.087402, -0.164062, … 0.005737]","[-1.0, 1.0, … -1.0]"
"[-0.310547, 0.087891, … -0.140625]","[-1.0, 1.0, … 1.0]"
"[-0.095703, -0.121094, … 0.029419]","[-1.0, 1.0, … -1.0]"
"[-0.134766, -0.152344, … 0.031738]","[1.0, -1.0, … 1.0]"
"[-0.28125, -0.302734, … -0.029297]","[-1.0, 1.0, … -1.0]"
